# The 2026 NeuroGolf Championship
## Zero-MAC Heuristics + Deep Convolutions & Progressive Pruning

Welcome to the ultimate code-golfing challenge for neural networks. In this competition,
**solving** the ARC-AGI transformation is only the entry requirement. The real objective
is solving it with the **smallest, cheapest network imaginable**.

---

### The Scoring Metric: Why Every MAC Counts

Our score for any correctly solved task is:

$$\text{score} = \max\!\left(1,\ 25 - \ln(\text{cost})\right)$$

where:

$$\text{cost} = N_\text{params} + N_\text{bytes} + N_\text{MACs}$$

Operating on 30×30 grids, a single standard $3 \times 3$ convolutional layer with 10
input and 10 output channels costs:

$$N_\text{params} = 10 \times 10 \times 3 \times 3 = 900$$
$$N_\text{MACs} = 10 \times 10 \times 3 \times 3 \times 30 \times 30 = 810{,}000$$

That $\ln(810{,}000 + \ldots) \approx 13.6$, immediately surrendering more than half
the maximum score. One standard convolution is catastrophic.

---

Let's start.

In [ ]:
import os
import json
import zipfile
import math
import copy
import sys
import time
import tempfile
import importlib.util
import numpy as np
from pathlib import Path
from itertools import product as iprod

# -- Visualisation --------------------------------------------------------------
import matplotlib.pyplot as plt
from matplotlib import colors

# -- ONNX stack -----------------------------------------------------------------
import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto

try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime -q")
    import onnxruntime as ort

try:
    import onnx_tool  # noqa: F401  (used for exact profiling if available)
except ImportError:
    os.system("pip install onnx-tool -q")

# ==============================================================================
# CONFIGURATION & ARC-AGI CONSTANTS
# ==============================================================================

#: Root directory that contains task001.json … task400.json
TASK_DIR = Path("/kaggle/input/competitions/neurogolf-2026")

#: Where ONNX files and the final submission.zip are written
OUTPUT_DIR = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
)

# One-hot encoding dimensionality: 10 colours × 30 rows × 30 cols
C, H, W = 10, 30, 30
HW  = H * W           # 900  - spatial positions on the flat canvas
CHW = C * H * W       # 9000 - total elements per sample

# Validate against EVERY arc-gen example (not just a small subset)
MAX_ARC_GEN_VALIDATE = 9_999

# When a Gather index has no legal source pixel, point it here:
# the last canvas position is permanently zero-padded, so it acts as a
# safe "black hole" that emits the background colour (channel 0 = 1, rest 0).
SAFE_PAD_IDX = HW - 1

# -- Matplotlib colour map matching the official ARC-AGI palette ----------------
ARC_CMAP = colors.ListedColormap([
    "#000000",  # 0  black
    "#0074D9",  # 1  blue
    "#FF4136",  # 2  red
    "#2ECC40",  # 3  green
    "#FFDC00",  # 4  yellow
    "#AAAAAA",  # 5  grey
    "#F012BE",  # 6  magenta
    "#FF851B",  # 7  orange
    "#7FDBFF",  # 8  azure
    "#870C25",  # 9  maroon
])
ARC_NORM = colors.Normalize(vmin=0, vmax=9)

print(
    f"Environment ready.  "
    f"Canvas: C={C}, H={H}, W={W}  |  "
    f"HW={HW}, CHW={CHW}  |  "
    f"SAFE_PAD_IDX={SAFE_PAD_IDX}"
)

# Cell 3 - Data Utilities and Cost Estimation

## Tensor Representation

Every ARC task presents grids as 2-D integer arrays of shape `(H, W)` with
values in `{0, …, 9}`. Our networks, however, operate in a **one-hot channel
space** of shape `(1, 10, 30, 30)`. The encoding is bijective:

```
channel c  is 1.0  at position (r, c_idx)   iff  grid[r, c_idx] == c
channel c  is 0.0  everywhere  outside the valid grid region
```

All channels are **simultaneously** zero in the padding zone (`r >= H_grid` or
`c_idx >= W_grid`), representing the "background" that lies outside the original
image border. Decoding reverses this with an `argmax` over the channel axis,
falling back to colour 0 wherever the maximum activation is below 0.5.

## Why Our Cost Estimator Differs from a Naive FLOP Counter

The official scoring formula counts:

```
cost = params  +  memory_bytes  +  MACs
```

Crucially, **only `Conv` and `Gemm`/`MatMul` nodes contribute MACs**. The
following operations are treated as free routing by the organisers:

| ONNX Op   | Description                   | MACs |
|-----------|-------------------------------|------|
| `Gather`  | Index-select along an axis    | 0    |
| `Slice`   | Contiguous sub-tensor extract | 0    |
| `Pad`     | Zero-pad a tensor             | 0    |
| `Transpose` | Dimension permutation       | 0    |
| `Tile`    | Repeat a tensor               | 0    |
| `Resize`  | Nearest-neighbour upsample    | 0    |
| `Add`     | Elementwise addition          | 0    |
| `Mul`     | Elementwise multiply          | 0    |
| `Max`     | Elementwise maximum           | 0    |
| `Relu`    | Elementwise rectification     | 0    |
| `Concat`  | Concatenation along an axis   | 0    |

This asymmetry is the **central insight** of our strategy. A `Gather` node that
permutes 900 spatial pixels costs almost nothing in the scoring metric - it
contributes only via the memory footprint of its index tensor (900 int32 values
= 3,600 bytes). Compare that to the 810,000 MACs a single standard $3 \times 3$
convolution would introduce.

Our `estimate_model_cost` function faithfully replicates this by iterating over
the ONNX graph and summing MACs **only** for `Conv` and `Gemm` nodes. When the
official `neurogolf_utils.score_network` utility is available on the Kaggle host,
we delegate to it for bit-perfect scores.

## Visualisation

The `plot_task` helper renders each train/test pair side-by-side using the
official ARC colour palette, which is invaluable when inspecting heuristic
detections and debugging edge cases in the pattern detectors.

In [ ]:
# ==============================================================================
# CELL 4 - DATA UTILITIES, VISUALISATION, AND COST ESTIMATION
# ==============================================================================


# -- I/O helpers ----------------------------------------------------------------

def load_task(path: Path) -> dict:
    """Load an ARC task JSON file and return its dict."""
    with open(path) as f:
        return json.load(f)


def grid_to_tensor(grid) -> np.ndarray:
    """
    Convert a 2-D integer ARC grid to a one-hot tensor of shape (1, C, H, W).

    Pixels outside the grid's native extents are left as all-zeros, which the
    network interprets as background (the 30×30 canvas padding zone).
    """
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    if h > H or w > W:
        return t
    for v in range(C):
        t[0, v, :h, :w] = (g == v).astype(np.float32)
    return t


def tensor_to_grid(t: np.ndarray, out_h: int, out_w: int) -> list:
    """
    Decode a (1, C, H, W) network output back to a 2-D integer list.

    Uses argmax over the channel axis; positions whose maximum activation
    is below 0.5 are assigned colour 0 (background).
    """
    arr = t[0]
    out_h, out_w = min(out_h, H), min(out_w, W)
    slice_ = arr[:, :out_h, :out_w]
    max_vals = slice_.max(axis=0)
    argmax   = slice_.argmax(axis=0)
    grid = np.where(max_vals < 0.5, 0, argmax).astype(int)
    return grid.tolist()


# -- Visualisation --------------------------------------------------------------

def plot_task(task_data: dict, task_name: str = "Task") -> None:
    """
    Render every train and test pair of an ARC task side-by-side using the
    official ARC colour palette.  Call this in a notebook cell to inspect
    which heuristic a given task might require.
    """
    train_pairs = task_data.get("train", [])
    test_pairs  = task_data.get("test",  [])
    all_pairs   = train_pairs + test_pairs
    n = len(all_pairs)

    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    fig.suptitle(f"{task_name}", fontsize=15, fontweight="bold")

    for i, pair in enumerate(all_pairs):
        prefix = "Train" if i < len(train_pairs) else "Test"
        ax_in  = axes[0, i] if n > 1 else axes[0]
        ax_out = axes[1, i] if n > 1 else axes[1]

        ax_in.imshow(pair["input"],  cmap=ARC_CMAP, norm=ARC_NORM)
        ax_in.set_title(f"{prefix} {i + 1}  In",  fontsize=9)
        ax_in.axis("off")

        ax_out.imshow(pair["output"], cmap=ARC_CMAP, norm=ARC_NORM)
        ax_out.set_title(f"{prefix} {i + 1}  Out", fontsize=9)
        ax_out.axis("off")

    plt.tight_layout()
    plt.show()


# -- Exact scoring via the host utility ----------------------------------------

_UTILS = None

def _load_utils():
    """Lazily import neurogolf_utils from the Kaggle data directory."""
    global _UTILS
    if _UTILS is not None:
        return _UTILS
    for d in [TASK_DIR, Path("/kaggle/input/competitions/neurogolf-2026")]:
        up = d / "neurogolf_utils" / "neurogolf_utils.py"
        if up.exists():
            spec = importlib.util.spec_from_file_location("ng", up)
            mod  = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            _UTILS = mod
            return mod
    return None


def exact_cost_via_utils(model) -> int | None:
    """
    Delegate to the official neurogolf_utils.score_network for a bit-perfect
    cost figure.  Returns None when the utility is unavailable.
    """
    utils = _load_utils()
    if utils is None:
        return None
    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
            f.write(model.SerializeToString())
            tmp_path = f.name
        macs, mem, params = utils.score_network(tmp_path)
        return int(macs + mem + params)
    except Exception:
        return None
    finally:
        if tmp_path:
            try:
                os.unlink(tmp_path)
            except OSError:
                pass


# -- Fallback cost estimator ----------------------------------------------------

def estimate_model_cost(model) -> int:
    """
    Parse the ONNX graph and compute:
        cost = total_params + total_bytes + total_MACs

    Only ``Conv`` and ``Gemm`` nodes contribute MACs.  All elementwise
    ops (Add, Mul, Max, Relu) and all routing ops (Gather, Slice, Pad,
    Transpose, Tile, Resize, Concat) cost exactly 0 MACs, matching the
    official scoring specification.

    When neurogolf_utils is available on the host, this function delegates
    to the exact scorer automatically.
    """
    ec = exact_cost_via_utils(model)
    if ec is not None:
        return ec

    total_params = 0
    total_bytes  = 0
    tensors: dict[str, np.ndarray] = {}

    # Accumulate all weight tensors from graph initialisers …
    for init in model.graph.initializer:
        arr = onh.to_array(init)
        tensors[init.name] = arr
        total_params += arr.size
        total_bytes  += arr.nbytes

    # … and from inline Constant nodes
    for node in model.graph.node:
        if node.op_type == "Constant":
            for attr in node.attribute:
                if attr.t and (
                    attr.t.raw_data
                    or list(attr.t.float_data)
                    or list(attr.t.int64_data)
                    or list(attr.t.int32_data)
                ):
                    arr = onh.to_array(attr.t)
                    if node.output:
                        tensors[node.output[0]] = arr
                    total_params += arr.size
                    total_bytes  += arr.nbytes

    # Count MACs only for Conv and Gemm
    total_macs = 0
    for node in model.graph.node:
        if node.op_type == "Conv":
            if len(node.input) >= 2 and node.input[1] in tensors:
                w = tensors[node.input[1]]
                if w.ndim == 4:
                    c_out, c_in, kh, kw = w.shape
                    # Standard counting: output positions × kernel volume × channels
                    total_macs += c_out * c_in * kh * kw * H * W
        elif node.op_type == "Gemm":
            if len(node.input) >= 2 and node.input[1] in tensors:
                w = tensors[node.input[1]]
                if w.ndim == 2:
                    total_macs += w.shape[0] * w.shape[1]

    return total_params + total_bytes + total_macs


# -- ONNX correctness checker --------------------------------------------------

def check_model_correct(model, pairs: list) -> bool:
    """
    Run *model* under ONNXRuntime against every (input, output) pair and
    return True only if all predictions match exactly.
    """
    if not pairs:
        return True
    try:
        buf = model.SerializeToString()
        so  = ort.SessionOptions()
        so.log_severity_level = 3  # suppress verbose runtime logs
        sess = ort.InferenceSession(
            buf, sess_options=so, providers=["CPUExecutionProvider"]
        )
        for p in pairs:
            inp      = grid_to_tensor(p["input"])
            out_g    = np.array(p["output"])
            oh_, ow_ = out_g.shape
            pred     = sess.run(None, {"input": inp})[0]
            if tensor_to_grid(pred, oh_, ow_) != p["output"]:
                return False
        return True
    except Exception:
        return False


print("Data utilities and cost estimator loaded.")

# Cell 5 - Phase 1: Zero-MAC Heuristics and ONNX Graph Compilation

## The Core Insight: Computation That Is Not Computation

The competition's scoring metric implicitly defines two classes of operations:

**Class A - Arithmetic (non-zero MACs):** `Conv`, `Gemm`, `MatMul`.
Every multiply-accumulate in these layers is taxed at 1 unit.

**Class B - Routing (zero MACs):** `Gather`, `Slice`, `Pad`, `Transpose`,
`Tile`, `Resize`, `Concat`, `Add`, `Mul`, `Max`, `Relu`.
These operations physically rearrange or scale data in memory but perform no
dot-product arithmetic.  The scoring formula assigns them zero MAC cost.

Our Phase 1 strategy **compiles** detected symbolic rules directly into Class B
graphs.  Because the compilation is exact and deterministic, correctness on the
private benchmark suite is guaranteed by construction - there is no learned
approximation that could fail to generalise.

---

## The ONNX Primitive Toolkit

### `Gather` - Spatial Permutation

`Gather` along a flattened spatial axis is the most powerful primitive.
Given a flat index tensor `gi` of shape `(HW,)`, it produces an output
where position `oi` is filled with the content of input position `gi[oi]`.

This single node can express any **bijective pixel permutation** - rotations,
reflections, transpositions, and arbitrary rearrangements - without a single
multiply.  The only cost is the `gi` array itself: 900 `int32` values = 3,600 bytes.

### `Slice` - Crop and Stride

`Slice` extracts a rectangular sub-tensor along any combination of axes.
With a step of `−1`, it **reverses** along that axis.  We use reversed `Slice`
to implement flips and 180° rotations without any `Gather` index tensor.

### `Pad` - Canvas Extension

`Pad` extends a tensor with zeros.  After a `Slice` crop or a `Transpose`,
we use `Pad` to restore the result to the required `(1, C, 30, 30)` output
shape, zero-filling the regions outside the active grid.

### `Transpose` - Axis Permutation

`Transpose` with `perm=[0, 1, 3, 2]` swaps the spatial axes (H and W),
performing a matrix transpose on every channel simultaneously.
Combined with a reversed `Slice` on one axis, this yields 90° rotations.

### `Tile` + `Resize` - Scale Operations

`Tile` repeats a tensor an integer number of times along any axes, implementing
pixel-perfect tiling and self-concatenation.  `Resize` with
`mode="nearest"` performs integer upscaling by nearest-neighbour interpolation.
Both are zero-MAC operations whose only cost is the scale-factor constant tensor.

---

## The Detection and Compilation Pipeline

For each task we run a battery of **detectors** - pure-NumPy functions that
inspect the raw grid arrays and return a detected parameter set (or `None`).
Detection is O(pairs × grid_size) and takes microseconds.

If a detector fires, we immediately **compile** the matching ONNX primitive and
run it through `check_model_correct` against the full combined dataset
(train + test + arc-gen).  If it passes, we record it as the current best model.

We always keep the **cheapest passing model** across all fired detectors, because
some tasks can be solved by multiple primitives (e.g., a 180° rotation is also a
double flip, which is also a spatial `Gather` permutation).  The cheapest among
these is typically the pure `Slice`/`Transpose` approach.

### Why We Try Both `Gather` and Cheap Slice/Pad Variants

A `Gather`-based rotation encodes 900 `int32` indices regardless of grid size.
For large grids that genuinely fill the 30×30 canvas, this is optimal.
For small grids (e.g., 3×3 patterns), the `Gather` tensor is mostly padding indices,
and a `Slice + Transpose + Pad` chain can be cheaper because it stores only a handful
of small constant scalars (start/end/step indices are single integers each).
We try both and keep the winner.

### Spatial Zero-Ambiguity Solver

Beyond the named detectors, we include a general-purpose spatial solver.
It constructs a dictionary mapping every unique column signature in the
input tensor to the spatial positions that carry it.  If every non-zero
output position maps to exactly one unambiguous input position, we build
a `Gather` index tensor directly and validate it.  This catches arbitrary
pixel permutations that no named detector covers.

### Spatial Relaxed Solver

A softer variant tolerates up to 12 ambiguous positions (multiple candidate
source pixels with the same colour signature) and resolves them by brute-force
enumeration of all combinations, validating each candidate.  This catches
symmetric layouts where a few positions have identical content in the
training examples.

In [ ]:
# ==============================================================================
# CELL 6 - ONNX BUILDERS AND PATTERN DETECTORS
# ==============================================================================


# -----------------------------------------------------------------------------
# SECTION A: Low-level ONNX node factories
# -----------------------------------------------------------------------------

def _const_node(name: str, arr: np.ndarray):
    """Create an ONNX Constant node that emits ``arr``."""
    t = onh.from_array(arr, name=name + "_v")
    return oh.make_node("Constant", [], [name], value=t)


def _slice_reverse_nodes(axes, dims, in_name, out_name, suf=""):
    """Build Slice nodes that reverse a tensor along the given axes."""
    starts = [d - 1 for d in dims]
    ends   = [-d - 1 for d in dims]
    steps  = [-1] * len(dims)
    return [
        _const_node(f"rs_s{suf}", np.asarray(starts, dtype=np.int64)),
        _const_node(f"rs_e{suf}", np.asarray(ends,   dtype=np.int64)),
        _const_node(f"rs_a{suf}", np.asarray(axes,   dtype=np.int64)),
        _const_node(f"rs_p{suf}", np.asarray(steps,  dtype=np.int64)),
        oh.make_node(
            "Slice",
            [in_name,
             f"rs_s{suf}", f"rs_e{suf}", f"rs_a{suf}", f"rs_p{suf}"],
            [out_name],
        ),
    ]


def _pad_to_canvas_nodes(in_name, out_name, out_h, out_w, suf=""):
    """Zero-pad a tensor from (1,C,out_h,out_w) to (1,C,H,W)."""
    pads = np.asarray(
        [0, 0, 0, 0, 0, 0, H - out_h, W - out_w], dtype=np.int64
    )
    return [
        _const_node(f"pd{suf}", pads),
        oh.make_node("Pad", [in_name, f"pd{suf}"], [out_name], mode="constant"),
    ]


def _crop_nodes(in_name, out_name, h, w, suf=""):
    """Crop the spatial dimensions of a (1,C,H,W) tensor to (1,C,h,w)."""
    return [
        _const_node(f"cs{suf}", np.asarray([0, 0], dtype=np.int64)),
        _const_node(f"ce{suf}", np.asarray([h, w], dtype=np.int64)),
        _const_node(f"ca{suf}", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", [in_name, f"cs{suf}", f"ce{suf}", f"ca{suf}"], [out_name]),
    ]


def _build_model_from_nodes(nodes: list, name: str):
    """Wrap a node list in a standard (1,C,H,W) -> (1,C,H,W) ONNX graph."""
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    graph = oh.make_graph(nodes, name, [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


# -----------------------------------------------------------------------------
# SECTION B: Named zero-MAC spatial transformers
# -----------------------------------------------------------------------------

def make_identity_onnx():
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    graph = oh.make_graph(
        [oh.make_node("Identity", ["input"], ["output"])], "identity", [X], [Y]
    )
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_cheap_rot180_framed(h, w):
    nodes  = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2, 3], [h, w], "content", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "rot180_framed")


def make_cheap_rot90cw_framed(h, w):
    nodes  = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([3], [h], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "rot90cw_framed")


def make_cheap_rot90ccw_framed(h, w):
    nodes  = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2], [w], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "rot90ccw_framed")


def make_cheap_flip_h_framed(h, w):
    nodes  = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2], [h], "content", "flipped", "r")
    nodes += _pad_to_canvas_nodes("flipped", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "flip_h_framed")


def make_cheap_flip_w_framed(h, w):
    nodes  = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([3], [w], "content", "flipped", "r")
    nodes += _pad_to_canvas_nodes("flipped", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "flip_w_framed")


def make_cheap_transpose_framed(h, w):
    nodes  = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _pad_to_canvas_nodes("t", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "transpose_framed")


def make_cheap_anti_transpose_framed(h, w):
    nodes  = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2, 3], [w, h], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "anti_transpose_framed")


def make_cheap_crop(r0, c0, out_h, out_w):
    nodes = [
        _const_node("cs", np.asarray([r0, c0],               dtype=np.int64)),
        _const_node("ce", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node("ca", np.asarray([2, 3],                  dtype=np.int64)),
        oh.make_node("Slice", ["input", "cs", "ce", "ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", out_h, out_w, "p")
    return _build_model_from_nodes(nodes, "crop_net")


def make_cheap_tile(tr, tc, in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [
        _const_node("tile_r", np.asarray([1, 1, tr, tc], dtype=np.int64)),
        oh.make_node("Tile", ["content", "tile_r"], ["tiled"]),
    ]
    nodes += _pad_to_canvas_nodes("tiled", "output", in_h * tr, in_w * tc, "p")
    return _build_model_from_nodes(nodes, "tile_net")


def make_cheap_upscale(f, in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [
        _const_node("rsz_roi",   np.asarray([], dtype=np.float32)),
        _const_node("rsz_scales", np.asarray([], dtype=np.float32)),
        _const_node("rsz_sizes",
                    np.asarray([1, C, in_h * f, in_w * f], dtype=np.int64)),
        oh.make_node(
            "Resize",
            ["content", "rsz_roi", "rsz_scales", "rsz_sizes"],
            ["upscaled"],
            mode="nearest",
            coordinate_transformation_mode="asymmetric",
            nearest_mode="floor",
        ),
    ]
    nodes += _pad_to_canvas_nodes("upscaled", "output", in_h * f, in_w * f, "p")
    return _build_model_from_nodes(nodes, "upscale_net")


def make_cheap_downscale(f, in_h, in_w):
    out_h, out_w = in_h // f, in_w // f
    nodes = [
        _const_node("ds_s", np.asarray([0, 0],       dtype=np.int64)),
        _const_node("ds_e", np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node("ds_a", np.asarray([2, 3],        dtype=np.int64)),
        _const_node("ds_p", np.asarray([f, f],        dtype=np.int64)),
        oh.make_node(
            "Slice",
            ["input", "ds_s", "ds_e", "ds_a", "ds_p"],
            ["downsampled"],
        ),
    ]
    nodes += _pad_to_canvas_nodes("downsampled", "output", out_h, out_w, "p")
    return _build_model_from_nodes(nodes, "downscale")


def make_cheap_framed_translation(grid_h, grid_w, dr, dc):
    src_r0 = max(0, -dr);  src_r1 = min(grid_h, grid_h - dr)
    src_c0 = max(0, -dc);  src_c1 = min(grid_w, grid_w - dc)
    dst_r0 = max(0,  dr);  dst_c0 = max(0,  dc)
    rh = src_r1 - src_r0;  rw = src_c1 - src_c0
    if rh <= 0 or rw <= 0:
        return None
    nodes = [
        _const_node("fts", np.asarray([src_r0, src_c0], dtype=np.int64)),
        _const_node("fte", np.asarray([src_r1, src_c1], dtype=np.int64)),
        _const_node("fta", np.asarray([2, 3],           dtype=np.int64)),
        oh.make_node("Slice", ["input", "fts", "fte", "fta"], ["fsliced"]),
    ]
    pads = np.asarray(
        [0, 0, dst_r0, dst_c0, 0, 0, H - dst_r0 - rh, W - dst_c0 - rw],
        dtype=np.int64,
    )
    nodes += [
        _const_node("fpp", pads),
        oh.make_node("Pad", ["fsliced", "fpp"], ["output"], mode="constant"),
    ]
    return _build_model_from_nodes(nodes, "framed_translate")


def make_cheap_concat_flip_w(in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([3], [in_w], "content", "flipped", "fw")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "concat_flip_w")


def make_cheap_concat_flip_h(in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2], [in_h], "content", "flipped", "fh")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=2)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h * 2, in_w, "p")
    return _build_model_from_nodes(nodes, "concat_flip_h")


def make_cheap_quadrant_mirror(in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([3], [in_w], "content", "fw", "fw")
    nodes += [oh.make_node("Concat", ["content", "fw"], ["top_row"], axis=3)]
    nodes += _slice_reverse_nodes([2], [in_h], "content", "fh", "fh")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "fr", "fr")
    nodes += [oh.make_node("Concat", ["fh", "fr"], ["bot_row"], axis=3)]
    nodes += [oh.make_node("Concat", ["top_row", "bot_row"], ["full"], axis=2)]
    nodes += _pad_to_canvas_nodes("full", "output", in_h * 2, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "quadrant_mirror")


def make_cheap_self_concat_h(in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [oh.make_node("Concat", ["content", "content"], ["concated"], axis=2)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h * 2, in_w, "p")
    return _build_model_from_nodes(nodes, "self_concat_h")


def make_cheap_self_concat_w(in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [oh.make_node("Concat", ["content", "content"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "self_concat_w")


def make_cheap_row_broadcast(in_h, in_w, row_idx=0):
    nodes = [
        _const_node("rb_s", np.asarray([row_idx],         dtype=np.int64)),
        _const_node("rb_e", np.asarray([row_idx + 1],     dtype=np.int64)),
        _const_node("rb_a", np.asarray([2],               dtype=np.int64)),
        oh.make_node("Slice", ["input", "rb_s", "rb_e", "rb_a"], ["one_row"]),
        _const_node("rb_r", np.asarray([1, 1, in_h, 1],   dtype=np.int64)),
        oh.make_node("Tile", ["one_row", "rb_r"], ["tiled"]),
        _const_node("rb_cs", np.asarray([0, 0],           dtype=np.int64)),
        _const_node("rb_ce", np.asarray([in_h, in_w],     dtype=np.int64)),
        _const_node("rb_ca", np.asarray([2, 3],           dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "rb_cs", "rb_ce", "rb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", in_h, in_w, "p")
    return _build_model_from_nodes(nodes, "row_broadcast")


def make_cheap_col_broadcast(in_h, in_w, col_idx=0):
    nodes = [
        _const_node("cb_s", np.asarray([col_idx],         dtype=np.int64)),
        _const_node("cb_e", np.asarray([col_idx + 1],     dtype=np.int64)),
        _const_node("cb_a", np.asarray([3],               dtype=np.int64)),
        oh.make_node("Slice", ["input", "cb_s", "cb_e", "cb_a"], ["one_col"]),
        _const_node("cb_r", np.asarray([1, 1, 1, in_w],   dtype=np.int64)),
        oh.make_node("Tile", ["one_col", "cb_r"], ["tiled"]),
        _const_node("cb_cs", np.asarray([0, 0],           dtype=np.int64)),
        _const_node("cb_ce", np.asarray([in_h, in_w],     dtype=np.int64)),
        _const_node("cb_ca", np.asarray([2, 3],           dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "cb_cs", "cb_ce", "cb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", in_h, in_w, "p")
    return _build_model_from_nodes(nodes, "col_broadcast")


def make_cheap_border_add(grid_h, grid_w, border, fill_color):
    oh_ = grid_h + 2 * border
    ow_ = grid_w + 2 * border
    if oh_ > H or ow_ > W:
        return None
    nodes  = _crop_nodes("input", "content", grid_h, grid_w, "pb0")
    nodes += [
        _const_node("pb_pads",
                    np.asarray([0, 0, border, border, 0, 0, border, border],
                               dtype=np.int64)),
        oh.make_node("Pad", ["content", "pb_pads"], ["padded"], mode="constant"),
    ]
    mask = np.zeros((1, C, oh_, ow_), dtype=np.float32)
    if 0 <= fill_color < C:
        mask[0, fill_color, :border,  :]       = 1.0
        mask[0, fill_color, -border:, :]        = 1.0
        mask[0, fill_color, :,        :border]  = 1.0
        mask[0, fill_color, :,        -border:] = 1.0
    nodes += [
        _const_node("pb_mask", mask),
        oh.make_node("Add", ["padded", "pb_mask"], ["bordered"]),
    ]
    nodes += _pad_to_canvas_nodes("bordered", "output", oh_, ow_, "pbf")
    return _build_model_from_nodes(nodes, "border_add")


def make_cheap_concat_rot180_w(in_h, in_w):
    nodes  = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "rotated", "r")
    nodes += [oh.make_node("Concat", ["content", "rotated"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "concat_rot180_w")


def make_cheap_crop_then_color(r0, c0, out_h, out_w, color_map):
    nodes = [
        _const_node("ctc_s", np.asarray([r0, c0],               dtype=np.int64)),
        _const_node("ctc_e", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node("ctc_a", np.asarray([2, 3],                  dtype=np.int64)),
        oh.make_node("Slice", ["input", "ctc_s", "ctc_e", "ctc_a"], ["cropped"]),
    ]
    inv = {d: s for s, d in color_map.items()}
    idx = np.arange(C, dtype=np.int64)
    mask = np.ones(C, dtype=np.float32)
    src_away = {s for s, d in color_map.items() if s != d}
    for d in range(C):
        if d in inv:
            idx[d] = inv[d]
        elif d in src_away:
            idx[d] = 0
            mask[d] = 0.0
    nodes += [_const_node("gi_ctc", idx)]
    if mask.min() == 1.0:
        nodes += [oh.make_node("Gather", ["cropped", "gi_ctc"], ["colored"], axis=1)]
    else:
        nodes += [
            oh.make_node("Gather", ["cropped", "gi_ctc"], ["g_ctc"], axis=1),
            _const_node("cm_ctc", mask.reshape(1, C, 1, 1)),
            oh.make_node("Mul", ["g_ctc", "cm_ctc"], ["colored"]),
        ]
    nodes += _pad_to_canvas_nodes("colored", "output", out_h, out_w, "pctc")
    return _build_model_from_nodes(nodes, "crop_color")


# -----------------------------------------------------------------------------
# SECTION C: Gather-based model builders
# -----------------------------------------------------------------------------

def _compact_mask(mask: np.ndarray) -> np.ndarray:
    """Broadcast a (1,C,1,1) mask to (1,1,1,1) when all channels are equal."""
    if (mask.ndim == 4 and mask.shape[0] == 1 and mask.shape[1] == C
            and np.all(mask[0, 0:1] == mask[0, :])):
        return mask[:, 0:1].copy()
    return mask


def make_gather_onnx(gather_indices: np.ndarray, mask=None):
    """
    Build a spatial-permutation network using a flat Gather over HW positions.

    ``gather_indices`` has shape (HW,) with dtype int32 or int64.
    Output position ``i`` receives the content of input position
    ``gather_indices[i]``.  Positions with index SAFE_PAD_IDX receive zeros.
    """
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["shape_chw"],
                     value=onh.from_array(np.array([1, C, HW], dtype=np.int64),
                                          name="shape_chw")),
        oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"]),
        oh.make_node("Constant", [], ["gather_idx"],
                     value=onh.from_array(gather_indices.astype(np.int32),
                                          name="gather_idx")),
        oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2),
        oh.make_node("Constant", [], ["out_shape"],
                     value=onh.from_array(np.array([1, C, H, W], dtype=np.int64),
                                          name="out_shape")),
    ]
    if mask is not None:
        mask = _compact_mask(mask)
        nodes += [
            oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"]),
            oh.make_node("Constant", [], ["mask"],
                         value=onh.from_array(mask.astype(np.float32), name="mask")),
            oh.make_node("Mul", ["gathered_4d", "mask"], ["output"]),
        ]
    else:
        nodes += [oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])]
    graph = oh.make_graph(nodes, "gather_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_channel_gather_onnx(gather_ch: np.ndarray):
    """Permute or duplicate channels using a single Gather along axis=1."""
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    graph = oh.make_graph(
        [
            oh.make_node("Constant", [], ["gi"],
                         value=onh.from_array(gather_ch.astype(np.int32), name="gi")),
            oh.make_node("Gather", ["input", "gi"], ["output"], axis=1),
        ],
        "ch_gather", [X], [Y],
    )
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_spatial_then_channel_gather_onnx(spatial_idx: np.ndarray,
                                          channel_idx: np.ndarray):
    """Compose a spatial Gather (flat HW) followed by a channel Gather."""
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["shape_chw"],
                     value=onh.from_array(np.array([1, C, HW], dtype=np.int64),
                                          name="shape_chw")),
        oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"]),
        oh.make_node("Constant", [], ["si"],
                     value=onh.from_array(spatial_idx.astype(np.int32), name="si")),
        oh.make_node("Gather", ["inp_chw", "si"], ["spatial_out"], axis=2),
        oh.make_node("Constant", [], ["out_shape"],
                     value=onh.from_array(np.array([1, C, H, W], dtype=np.int64),
                                          name="out_shape")),
        oh.make_node("Reshape", ["spatial_out", "out_shape"], ["spatial_4d"]),
        oh.make_node("Constant", [], ["ci"],
                     value=onh.from_array(channel_idx.astype(np.int32), name="ci")),
        oh.make_node("Gather", ["spatial_4d", "ci"], ["output"], axis=1),
    ]
    graph = oh.make_graph(nodes, "spatial_ch_gather", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_gather_then_conv1x1_onnx(gather_indices: np.ndarray,
                                   conv_weight: np.ndarray,
                                   mask=None):
    """Spatial Gather followed by a 1×1 colour-mix convolution."""
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["shape_chw"],
                     value=onh.from_array(np.array([1, C, HW], dtype=np.int64),
                                          name="shape_chw")),
        oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"]),
        oh.make_node("Constant", [], ["gather_idx"],
                     value=onh.from_array(gather_indices.astype(np.int32),
                                          name="gather_idx")),
        oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2),
        oh.make_node("Constant", [], ["out_shape"],
                     value=onh.from_array(np.array([1, C, H, W], dtype=np.int64),
                                          name="out_shape")),
        oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"]),
    ]
    w_tensor = onh.from_array(conv_weight.astype(np.float32), name="conv_weight")
    if mask is not None:
        mask = _compact_mask(mask)
        nodes += [
            oh.make_node("Constant", [], ["mask"],
                         value=onh.from_array(mask.astype(np.float32), name="mask")),
            oh.make_node("Mul", ["gathered_4d", "mask"], ["masked"]),
            oh.make_node("Constant", [], ["conv_weight"], value=w_tensor),
            oh.make_node("Conv", ["masked", "conv_weight"], ["output"],
                         kernel_shape=[1, 1], pads=[0, 0, 0, 0]),
        ]
    else:
        nodes += [
            oh.make_node("Constant", [], ["conv_weight"], value=w_tensor),
            oh.make_node("Conv", ["gathered_4d", "conv_weight"], ["output"],
                         kernel_shape=[1, 1], pads=[0, 0, 0, 0]),
        ]
    graph = oh.make_graph(nodes, "gather_conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_conv1x1_onnx(weight: np.ndarray):
    """Pure 1×1 colour-permutation convolution (no spatial work)."""
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["conv_weight"],
                     value=onh.from_array(weight.astype(np.float32),
                                          name="conv_weight")),
        oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                     kernel_shape=[1, 1], pads=[0, 0, 0, 0]),
    ]
    graph = oh.make_graph(nodes, "conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_const_onnx(const_output: np.ndarray):
    """Network that ignores the input and always emits a fixed tensor."""
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["const_out"],
                     value=onh.from_array(const_output.astype(np.float32),
                                          name="const_out")),
        oh.make_node("Identity", ["const_out"], ["output"]),
    ]
    graph = oh.make_graph(nodes, "const_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_symmetry_completion_onnx(gather_index_list: list, mask=None):
    """
    Max-pool multiple Gather views of the input to complete a symmetry.

    Each element of ``gather_index_list`` is a flat (HW,) index array
    corresponding to one symmetry axis.  The final output is the elementwise
    Max across all gathered views, optionally multiplied by a channel mask.
    """
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    ch_mask = np.ones((1, C, 1, 1), dtype=np.float32)
    ch_mask[0, 0, 0, 0] = 0.0   # suppress background channel bleed
    nodes = [
        oh.make_node("Constant", [], ["ch_mask"],
                     value=onh.from_array(ch_mask, name="ch_mask")),
        oh.make_node("Mul", ["input", "ch_mask"], ["input_nobg"]),
        oh.make_node("Constant", [], ["shape_chw"],
                     value=onh.from_array(np.array([1, C, HW], dtype=np.int64),
                                          name="shape_chw")),
        oh.make_node("Reshape", ["input_nobg", "shape_chw"], ["inp_chw"]),
        oh.make_node("Constant", [], ["out_shape"],
                     value=onh.from_array(np.array([1, C, H, W], dtype=np.int64),
                                          name="out_shape")),
    ]
    reshape_names = []
    for i, gi in enumerate(gather_index_list):
        gn, rn = f"gi_{i}", f"r_{i}"
        nodes += [
            oh.make_node("Constant", [], [gn],
                         value=onh.from_array(gi.astype(np.int32), name=gn)),
            oh.make_node("Gather", ["inp_chw", gn], [f"g_{i}"], axis=2),
            oh.make_node("Reshape", [f"g_{i}", "out_shape"], [rn]),
        ]
        reshape_names.append(rn)

    if len(reshape_names) == 1:
        nodes.append(oh.make_node("Identity", [reshape_names[0]], ["output"]))
    else:
        prev = reshape_names[0]
        for i in range(1, len(reshape_names)):
            is_last = i == len(reshape_names) - 1
            out = "output" if (is_last and mask is None) else f"max_{i}"
            nodes.append(oh.make_node("Max", [prev, reshape_names[i]], [out]))
            prev = out

    if mask is not None:
        mask = _compact_mask(mask)
        nodes += [
            oh.make_node("Constant", [], ["mask"],
                         value=onh.from_array(mask.astype(np.float32), name="mask")),
            oh.make_node("Mul", [prev, "mask"], ["output"]),
        ]
    graph = oh.make_graph(nodes, "sym_complete", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


# -----------------------------------------------------------------------------
# SECTION D: Gather index generators
# -----------------------------------------------------------------------------

def build_rotation_gather(angle: int, ih: int, iw: int) -> np.ndarray:
    k   = angle // 90
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    if k == 1:
        for r in range(iw):
            for c in range(ih):
                idx[r * W + c] = c * W + (iw - 1 - r)
    elif k == 2:
        for r in range(ih):
            for c in range(iw):
                idx[r * W + c] = (ih - 1 - r) * W + (iw - 1 - c)
    elif k == 3:
        for r in range(iw):
            for c in range(ih):
                idx[r * W + c] = (ih - 1 - c) * W + r
    return idx


def build_flip_gather(direction: str, ih: int, iw: int) -> np.ndarray:
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            if direction == "vertical":
                idx[r * W + c] = (ih - 1 - r) * W + c
            elif direction == "horizontal":
                idx[r * W + c] = r * W + (iw - 1 - c)
            else:
                idx[r * W + c] = (ih - 1 - r) * W + (iw - 1 - c)
    return idx


def build_h_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m  = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r * W + c] = r * W + c
            idx_m[r  * W + c] = r * W + (iw - 1 - c)
    return [idx_id, idx_m]


def build_v_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m  = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r * W + c] = r * W + c
            idx_m[r  * W + c] = (ih - 1 - r) * W + c
    return [idx_id, idx_m]


def build_both_sym_complete_gathers(ih, iw):
    idxs = []
    for t in range(4):
        idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
        for r in range(ih):
            for c in range(iw):
                if   t == 0: sr, sc = r,          c
                elif t == 1: sr, sc = r,          iw - 1 - c
                elif t == 2: sr, sc = ih - 1 - r, c
                else:        sr, sc = ih - 1 - r, iw - 1 - c
                idx[r * W + c] = sr * W + sc
        idxs.append(idx)
    return idxs


def build_diag_sym_complete_gathers(ih, iw):
    if ih != iw:
        return None
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m  = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r * W + c] = r * W + c
            idx_m[r  * W + c] = c * W + r
    return [idx_id, idx_m]


# -----------------------------------------------------------------------------
# SECTION E: Colour-map weight builders
# -----------------------------------------------------------------------------

def can_use_channel_gather_for_cmap(cmap, train_pairs) -> bool:
    """Return True when the colour map is a pure channel permutation (no mixing)."""
    if cmap is None:
        return False
    changed = {s: d for s, d in cmap.items() if s != d}
    if not changed:
        return False
    return set(changed.keys()) == set(changed.values())


def build_channel_gather_indices(cmap) -> np.ndarray:
    gi = np.arange(C, dtype=np.int32)
    for src, dst in cmap.items():
        if src != dst:
            gi[dst] = src
    return gi


def build_color_map_weight(color_map) -> np.ndarray:
    w = np.zeros((C, C, 1, 1), dtype=np.float32)
    for src, dst in color_map.items():
        if 0 <= src < C and 0 <= dst < C:
            w[dst, src, 0, 0] = 1.0
    for ch in range(C):
        if ch not in color_map:
            w[ch, ch, 0, 0] = 1.0
    return w


def build_nonzero_recolor_weight(tc: int) -> np.ndarray:
    """All non-background pixels -> colour ``tc``."""
    w = np.zeros((C, C, 1, 1), dtype=np.float32)
    w[0,  0, 0, 0] = 1.0
    for s in range(1, C):
        w[tc, s, 0, 0] = 1.0
    return w


def build_bg_recolor_weight(tc: int) -> np.ndarray:
    """Background pixels -> colour ``tc``; foreground unchanged."""
    w = np.zeros((C, C, 1, 1), dtype=np.float32)
    w[tc, 0, 0, 0] = 1.0
    for s in range(1, C):
        w[s, s, 0, 0] = 1.0
    return w


# -----------------------------------------------------------------------------
# SECTION F: Pattern detectors
# -----------------------------------------------------------------------------

def shapes_match(p) -> bool:
    return np.array(p["input"]).shape == np.array(p["output"]).shape


def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        if all(
            np.array_equal(np.rot90(np.array(p["input"]), k), np.array(p["output"]))
            for p in pairs
        ):
            return angle
    return None


def detect_flip(pairs):
    for axis_val, name in [(0, "vertical"), (1, "horizontal"), (-1, "both")]:
        def do_flip(ig, a=axis_val):
            return (np.flip(np.flip(ig, 0), 1) if a == -1
                    else np.flip(ig, axis=a))
        if all(np.array_equal(do_flip(np.array(p["input"])),
                              np.array(p["output"])) for p in pairs):
            return name
    return None


def detect_transpose(pairs) -> bool:
    return all(
        np.array_equal(np.array(p["input"]).T, np.array(p["output"]))
        for p in pairs
    )


def detect_anti_transpose(pairs) -> bool:
    return all(
        np.array_equal(
            np.flip(np.flip(np.array(p["input"]).T, 0), 1),
            np.array(p["output"]),
        )
        for p in pairs
    )


def detect_crop(pairs):
    results = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape;          oh_, ow_ = og.shape
        if oh_ > ih or ow_ > iw:
            return None
        found = False
        for r in range(ih - oh_ + 1):
            for c in range(iw - ow_ + 1):
                if np.array_equal(ig[r:r + oh_, c:c + ow_], og):
                    results.append((r, c, oh_, ow_))
                    found = True
                    break
            if found:
                break
        if not found:
            return None
    if len({(r[0], r[1]) for r in results}) == 1:
        return results[0]
    return None


def detect_tile(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape;          oh_, ow_ = og.shape
        if oh_ % ih or ow_ % iw:
            return None
        tr, tc = oh_ // ih, ow_ // iw
        if not np.array_equal(np.tile(ig, (tr, tc)), og):
            return None
    p0 = pairs[0]
    ig0, og0 = np.array(p0["input"]), np.array(p0["output"])
    return og0.shape[0] // ig0.shape[0], og0.shape[1] // ig0.shape[1]


def detect_scale(pairs):
    for factor in [2, 3, 4, 5]:
        if all(
            np.array(p["output"]).shape == (
                np.array(p["input"]).shape[0] * factor,
                np.array(p["input"]).shape[1] * factor,
            ) and np.array_equal(
                np.repeat(np.repeat(np.array(p["input"]), factor, 0), factor, 1),
                np.array(p["output"]),
            )
            for p in pairs
        ):
            return factor
    return None


def detect_color_replace(pairs):
    if not all(shapes_match(p) for p in pairs):
        return None
    replacements = {}
    for p in pairs:
        for a, b in zip(np.array(p["input"]).flat, np.array(p["output"]).flat):
            if a != b:
                if a in replacements and replacements[a] != b:
                    return None
                replacements[a] = b
    if not replacements:
        return None
    for p in pairs:
        ig = np.array(p["input"]); exp = ig.copy()
        for s, d in replacements.items():
            exp[ig == s] = d
        if not np.array_equal(exp, np.array(p["output"])):
            return None
    return replacements


def detect_color_swap(pairs):
    if not all(shapes_match(p) for p in pairs):
        return None
    swap_pairs: set = set()
    for p in pairs:
        for a, b in zip(np.array(p["input"]).flat, np.array(p["output"]).flat):
            if a != b:
                swap_pairs.add((min(a, b), max(a, b)))
    if len(swap_pairs) != 1:
        return None
    a, b = list(swap_pairs)[0]
    saw_ab = saw_ba = False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        exp = ig.copy(); exp[ig == a] = b; exp[ig == b] = a
        if not np.array_equal(exp, og):
            return None
        if np.any((ig == a) & (og == b)):
            saw_ab = True
        if np.any((ig == b) & (og == a)):
            saw_ba = True
    return (a, b) if saw_ab and saw_ba else None


def detect_pixel_permutation(pairs):
    if not all(shapes_match(p) for p in pairs):
        return None
    if len({np.array(p["input"]).shape for p in pairs}) != 1:
        return None
    ih, iw = np.array(pairs[0]["input"]).shape
    if ih > H or iw > W:
        return None
    all_in  = [np.array(p["input"]).flatten()  for p in pairs]
    all_out = [np.array(p["output"]).flatten() for p in pairs]
    assignment = {}
    for dst_idx in range(ih * iw):
        targets = [all_out[pi][dst_idx] for pi in range(len(pairs))]
        candidates = None
        for pi in range(len(pairs)):
            match = set(np.where(all_in[pi] == targets[pi])[0].tolist())
            candidates = match if candidates is None else candidates & match
            if not candidates:
                return None
        if len(candidates) != 1:
            return None
        assignment[dst_idx] = candidates.pop()
    gi = np.arange(HW, dtype=np.int64)
    for dst, src in assignment.items():
        dr, dc = divmod(dst, iw); sr, sc = divmod(src, iw)
        gi[dr * W + dc] = sr * W + sc
    for p in pairs:
        ig = np.array(p["input"]); padded = np.zeros((H, W), dtype=np.int32)
        padded[:ih, :iw] = ig
        pred = padded.flatten()[gi].reshape(H, W)[:ih, :iw]
        if not np.array_equal(pred, np.array(p["output"])):
            return None
    return gi


def detect_mirror_h_concat(pairs) -> bool:
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0], ig.shape[1] * 2): return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 1)], 1), og): return False
    return True


def detect_mirror_v_concat(pairs) -> bool:
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0] * 2, ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 0)], 0), og): return False
    return True


def detect_self_concat_h(pairs) -> bool:
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0], ig.shape[1] * 2): return False
        if not np.array_equal(np.concatenate([ig, ig], 1), og): return False
    return True


def detect_self_concat_v(pairs) -> bool:
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0] * 2, ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig, ig], 0), og): return False
    return True


def detect_quad_mirror(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape
        if og.shape != (ih * 2, iw * 2): return None
        exp = np.block([
            [ig,             np.flip(ig, 1)],
            [np.flip(ig, 0), np.flip(np.flip(ig, 0), 1)],
        ])
        if not np.array_equal(exp, og): return None
    return "tl"


def detect_h_symmetry_complete(pairs) -> bool:
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.array_equal(np.where(ig != 0, ig, np.flip(ig, 1)), og):
            return False
    return True


def detect_v_symmetry_complete(pairs) -> bool:
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.array_equal(np.where(ig != 0, ig, np.flip(ig, 0)), og):
            return False
    return True


def detect_both_symmetry_complete(pairs) -> bool:
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        stack = np.stack([
            ig,
            np.flip(ig, 1),
            np.flip(ig, 0),
            np.flip(np.flip(ig, 0), 1),
        ], 0)
        if not np.array_equal(np.where(ig != 0, ig, stack.max(0)), og):
            return False
    return True


def detect_diag_symmetry_complete(pairs) -> bool:
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] != ig.shape[1]: return False
        if not np.array_equal(np.where(ig != 0, ig, ig.T), og):
            return False
    return True


def detect_translation(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    ih, iw = np.array(pairs[0]["input"]).shape
    for dr in range(-min(ih, 11) + 1, min(ih, 11)):
        for dc in range(-min(iw, 11) + 1, min(iw, 11)):
            if dr == 0 and dc == 0: continue
            ok = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                hh, ww = ig.shape; s = np.zeros_like(ig)
                sr0 = max(0, -dr); sr1 = min(hh, hh - dr)
                sc0 = max(0, -dc); sc1 = min(ww, ww - dc)
                dr0 = max(0,  dr); dc0 = max(0,  dc)
                rh  = sr1 - sr0;  rw  = sc1 - sc0
                if rh > 0 and rw > 0:
                    s[dr0:dr0 + rh, dc0:dc0 + rw] = ig[sr0:sr1, sc0:sc1]
                if not np.array_equal(s, og): ok = False; break
            if ok: return (dr, dc)
    return None


def detect_downscale(pairs):
    for f in [2, 3, 4, 5]:
        if all(
            np.array(p["input"]).shape == (
                np.array(p["output"]).shape[0] * f,
                np.array(p["output"]).shape[1] * f,
            ) and np.array_equal(np.array(p["input"])[::f, ::f],
                                  np.array(p["output"]))
            for p in pairs
        ):
            return f
    return None


def detect_color_filter(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    cand = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        diff = (ig != og)
        if not np.any(diff): continue
        if not np.all(og[diff] == 0): return None
        local = {int(x) for x in ig[diff]}
        cand  = local if cand is None else cand | local
    if not cand: return None
    for p in pairs:
        ig = np.array(p["input"]).copy()
        for c_ in cand: ig[ig == c_] = 0
        if not np.array_equal(ig, np.array(p["output"])): return None
    return cand


def detect_row_broadcast(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    chosen = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        m = next((r for r in range(ig.shape[0]) if np.all(og == ig[r, :])), None)
        if m is None: return None
        if chosen is None: chosen = m
        elif chosen != m:  return None
    return chosen


def detect_col_broadcast(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    chosen = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        m = next((c for c in range(ig.shape[1])
                  if np.all(og == ig[:, c:c + 1])), None)
        if m is None: return None
        if chosen is None: chosen = m
        elif chosen != m:  return None
    return chosen


def detect_border_add(pairs):
    if not pairs: return None
    fi = np.array(pairs[0]["input"]); fo = np.array(pairs[0]["output"])
    ih, iw = fi.shape;               oh_, ow_ = fo.shape
    if oh_ - ih != ow_ - iw or oh_ <= ih: return None
    d = oh_ - ih
    if d % 2: return None
    b = d // 2
    if not np.array_equal(fo[b:b + ih, b:b + iw], fi): return None
    vals = (set(fo[:b, :].flat) | set(fo[-b:, :].flat)
            | set(fo[:, :b].flat) | set(fo[:, -b:].flat))
    if len(vals) != 1: return None
    fill = int(vals.pop())
    for p in pairs[1:]:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0] + 2 * b, ig.shape[1] + 2 * b): return None
        if not np.array_equal(og[b:b + ig.shape[0], b:b + ig.shape[1]], ig):
            return None
        mm = np.zeros(og.shape, bool)
        mm[:b, :] = mm[-b:, :] = mm[:, :b] = mm[:, -b:] = True
        if not np.all(og[mm] == fill): return None
    return (b, fill)


def detect_crop_plus_color(pairs):
    if not pairs: return None
    fi = np.array(pairs[0]["input"]); fo = np.array(pairs[0]["output"])
    oh_, ow_ = fo.shape; ih, iw = fi.shape
    if oh_ > ih or ow_ > iw: return None
    for r0 in range(ih - oh_ + 1):
        for c0 in range(iw - ow_ + 1):
            cm: dict = {}; ok = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                if og.shape != (oh_, ow_) or r0 + oh_ > ig.shape[0] or c0 + ow_ > ig.shape[1]:
                    ok = False; break
                for a, bv in zip(ig[r0:r0 + oh_, c0:c0 + ow_].flat, og.flat):
                    a, bv = int(a), int(bv)
                    if a in cm and cm[a] != bv: ok = False; break
                    cm[a] = bv
                if not ok: break
            if not ok: continue
            if all(k == v for k, v in cm.items()): continue
            valid = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                reg = ig[r0:r0 + oh_, c0:c0 + ow_].copy()
                for k2, v2 in cm.items(): reg[reg == k2] = v2
                if not np.array_equal(reg, og): valid = False; break
            if valid: return (r0, c0, oh_, ow_, cm)
    return None


def detect_concat_rot180_w(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0], ig.shape[1] * 2): return None
        if not np.array_equal(np.concatenate([ig, np.rot90(ig, 2)], 1), og):
            return None
    return True


def detect_nonzero_recolor(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    tc = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.all(og[ig == 0] == 0): return None
        nb = og[ig != 0]
        if nb.size == 0: continue
        u = np.unique(nb)
        if len(u) != 1: return None
        if tc is None: tc = u[0]
        elif tc != u[0]: return None
    return tc


def detect_bg_recolor(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    tc = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.array_equal(og[ig != 0], ig[ig != 0]): return None
        bo = og[ig == 0]
        if bo.size == 0: continue
        u = np.unique(bo)
        if len(u) != 1: return None
        if tc is None: tc = u[0]
        elif tc != u[0]: return None
    return tc


# -----------------------------------------------------------------------------
# SECTION G: General spatial solvers (fallback when named detectors miss)
# -----------------------------------------------------------------------------

def try_spatial_zero_ambiguity(all_pairs):
    """
    Build a Gather model by matching every output column signature to a unique
    input column.  Succeeds only when the mapping is 100% unambiguous.
    """
    for p in all_pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] > H or ig.shape[1] > W or og.shape[0] > H or og.shape[1] > W:
            return None

    inputs  = np.stack([grid_to_tensor(p["input"])[0].reshape(C, -1)  for p in all_pairs])
    outputs = np.stack([grid_to_tensor(p["output"])[0].reshape(C, -1) for p in all_pairs])

    lookup: dict = {}
    for si in range(HW):
        lookup.setdefault(inputs[:, :, si].tobytes(), []).append(si)

    gi = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for oi in range(HW):
        sa = outputs[:, :, oi]
        if np.all(sa == 0): continue
        cands = lookup.get(sa.tobytes())
        if not cands or len(cands) > 1: return None
        gi[oi] = cands[0]
    return make_gather_onnx(gi)


def try_spatial_relaxed(all_pairs, max_ambig: int = 12):
    """
    Like ``try_spatial_zero_ambiguity`` but tolerates up to ``max_ambig``
    positions with multiple candidate sources; resolves them by exhaustive
    search.
    """
    for p in all_pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] > H or ig.shape[1] > W or og.shape[0] > H or og.shape[1] > W:
            return None

    inputs  = np.stack([grid_to_tensor(p["input"])[0].reshape(C, -1)  for p in all_pairs])
    outputs = np.stack([grid_to_tensor(p["output"])[0].reshape(C, -1) for p in all_pairs])

    lookup: dict = {}
    for si in range(HW):
        lookup.setdefault(inputs[:, :, si].tobytes(), []).append(si)

    gi    = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    ambig = []
    for oi in range(HW):
        sa = outputs[:, :, oi]
        if np.all(sa == 0): continue
        cands = lookup.get(sa.tobytes())
        if not cands: return None
        gi[oi] = cands[0]
        if len(cands) > 1: ambig.append((oi, cands))

    if len(ambig) > max_ambig: return None

    if not ambig:
        m = make_gather_onnx(gi)
        return m if check_model_correct(m, all_pairs) else None

    m = make_gather_onnx(gi)
    if check_model_correct(m, all_pairs): return m

    if len(ambig) > 8: return None
    for combo in iprod(*(c for _, c in ambig)):
        for idx, (oi, _) in enumerate(ambig):
            gi[oi] = combo[idx]
        m = make_gather_onnx(gi)
        if check_model_correct(m, all_pairs): return m
    return None


print("ONNX builders and pattern detectors loaded.")

# Cell 7 - Phase 2: The Neural Waterfall and Sparse L1 Regularization

When no zero-MAC compilation succeeds, we must resort to learned networks.
The Neural Waterfall is a **strict cost hierarchy** of three PyTorch models,
each tried in sequence from cheapest to most expensive.  As soon as any model
validates correctly across the full combined dataset, we accept it and skip
the remaining (more expensive) options.

---

## Tier 1 - Standard Single-Layer Convolutions

After Zero-MAC heuristics fail, we abandon MAC economy in favor of raw representation power. We start with standard Conv2d layers (kernel sizes 1×1, 3×3, and 5×5) to handle color mixing and spatial transformations.

## Tier 2 - Two-Layer ReLU Network

For tasks involving nonlinear interactions (e.g., context-dependent color changes), a single linear convolution is insufficient. We escalate to a two-layer network:

$\mathbf{h} = \operatorname{ReLU}\!\left(W_1 * X + b_1\right), \quad \hat{Y} = W_2 * \mathbf{h}$

## Tier 3 - Three-Layer Conv Network

For the hardest tasks that require deeper spatial reasoning, we utilize a three-layer cascade. While this heavily penalizes our MAC score (often reaching ~846,000 MACs), securing the exact solution is prioritized over efficiency at this stage.

## Overcoming ARC-Gen Overfitting: Spatial Dropout

A major challenge in ARC is that tasks often contain only 2 to 4 training pairs. Deep networks (like our multi-layer tiers) tend to memorize exact coordinate locations rather than learning the underlying geometric rules. This causes them to fail against the unseen `arc-gen` validation tasks.

To counter this, we inject Spatial Noise (`nn.Dropout(p=0.1)`) at the input layer of our multi-layer networks. By randomly blinding the network to 10% of the grid pixels during training, we force the convolutions to learn robust spatial logic (e.g., "fill enclosed shapes") rather than memorizing the training backgrounds.

## Maximizing the Score: Progressive Pruning Search

The competition scoring metric heavily penalizes parameter count and file size: `25 - ln(cost)`.
While standard convolutions guarantee we have the capacity to solve complex tasks, they generate expensive dense weight matrices. Instead of applying a rigid mathematical penalty during training, which often causes underfitting, we use a **Progressive Pruning Search** after convergence.

Before exporting to ONNX, we test the network against a cascade of aggressive hard-pruning thresholds: `[0.2, 0.1, 0.05, 0.01, 0.0]`. We keep the sparsest possible tensor that still achieves 100% accuracy on the validation pairs. This allows the ONNX compiler to optimize away explicit zeros, dramatically dropping our final MAC and parameter cost without sacrificing correct answers.

In [ ]:
# ==============================================================================
# CELL 8 - NEURAL WATERFALL: PYTORCH FALLBACK SOLVERS
# ==============================================================================

try:
    import torch
    import torch.nn as nn
    _TORCH_OK = True
except ImportError:
    _TORCH_OK = False
    print("WARNING: PyTorch not found.  Neural fallbacks will be disabled.")


def _pairs_to_tensors(pairs):
    """Stack ARC pairs into (N, C, H, W) PyTorch tensors."""
    inp = torch.tensor(
        np.stack([grid_to_tensor(p["input"])[0]  for p in pairs]),
        dtype=torch.float32,
    )
    out = torch.tensor(
        np.stack([grid_to_tensor(p["output"])[0] for p in pairs]),
        dtype=torch.float32,
    )
    return inp, out


def _prune(arr: np.ndarray, thresh: float = 0.02) -> np.ndarray:
    """Hard-zero weights below ``thresh`` to induce true sparsity."""
    return np.where(np.abs(arr) < thresh, 0.0, arr)

# -----------------------------------------------------------------------------
# TIER 1: Standard Single-Layer Convolution
# -----------------------------------------------------------------------------

def _make_conv_onnx_from_weights(w: np.ndarray, b, kernel_size: int):
    """Build a single-Conv ONNX graph from numpy weight and optional bias."""
    pad = kernel_size // 2
    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["w"],
                     value=onh.from_array(w.astype(np.float32), name="w")),
    ]
    if b is not None:
        nodes += [
            oh.make_node("Constant", [], ["b"],
                         value=onh.from_array(b.astype(np.float32), name="b")),
            oh.make_node("Conv", ["input", "w", "b"], ["output"],
                         kernel_shape=[kernel_size, kernel_size],
                         pads=[pad] * 4),
        ]
    else:
        nodes += [
            oh.make_node("Conv", ["input", "w"], ["output"],
                         kernel_shape=[kernel_size, kernel_size],
                         pads=[pad] * 4),
        ]
    m = oh.make_model(
        oh.make_graph(nodes, "std_conv", [X], [Y]),
        opset_imports=[oh.make_opsetid("", 17)],
    )
    m.ir_version = 8
    return m


def try_standard_conv_fast(pairs, all_pairs,
                           kernel_size: int = 1,
                           max_steps:   int = 1_500,
                           lr:          float = 0.02,
                           use_bias:    bool = False):
    """
    Train a standard single-layer Conv2d and export the best (pruned/rounded)
    variant to ONNX.

    We evaluate three weight post-processing strategies and keep whichever
    produces the correct output at the lowest cost:
        1. Rounded weights  (most aggressive compression for colour tasks)
        2. L1-pruned weights (moderate sparsity)
        3. Raw weights      (fallback if rounding or pruning breaks correctness)
    """
    if not _TORCH_OK:
        return None
    inp, out = _pairs_to_tensors(pairs)
    pad  = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=use_bias)

    nn.init.zeros_(conv.weight)
    if use_bias:
        nn.init.zeros_(conv.bias)
    for i in range(C):
        conv.weight.data[i, i, pad, pad] = 1.0

    optimizer = torch.optim.Adam(conv.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

    best_loss  = float("inf")
    best_state = None
    plateau    = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(conv(inp), out)
        loss.backward()
        optimizer.step()
        scheduler.step()

        if loss.item() < best_loss - 1e-7:
            best_loss  = loss.item()
            best_state = copy.deepcopy(conv.state_dict())
            plateau    = 0
        else:
            plateau += 1

        if step == 250  and best_loss > 0.30: return None
        if step == 600  and best_loss > 0.10: return None
        if plateau > 400 and best_loss > 0.01: return None
        if best_loss < 1e-6: break

    if best_loss > 0.05:
        return None

    conv.load_state_dict(best_state)
    w = conv.weight.detach().numpy()
    b = conv.bias.detach().numpy() if use_bias else None

    best_model = None
    best_cost  = float("inf")
    b_try = np.round(b) if use_bias else None
    
    # Progressive Pruning Search: Try aggressive sparsification first
    thresholds = [0.2, 0.1, 0.05, 0.01, 0.0]
    for thresh in thresholds:
        w_try = np.where(np.abs(w) < thresh, 0.0, w)
        m = _make_conv_onnx_from_weights(w_try, b_try, kernel_size)
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost:
                best_cost, best_model = c, m

    # Rounding Wildcard: Excellent for pure colour-replacement tasks
    w_round = np.round(w)
    m_round = _make_conv_onnx_from_weights(w_round, b_try, kernel_size)
    if check_model_correct(m_round, all_pairs):
        c = estimate_model_cost(m_round)
        if c < best_cost:
            best_cost, best_model = c, m_round

    return best_model


# -----------------------------------------------------------------------------
# TIER 2: Two-Layer ReLU Network
# -----------------------------------------------------------------------------

def try_two_layer_conv_fast(pairs, all_pairs,
                            ks1:       int = 3,
                            ks2:       int = 1,
                            hidden:    int = 16,
                            max_steps: int = 2_500,
                            lr:        float = 0.015):
    """
    Train a Conv -> ReLU -> Conv network and export to ONNX.

    This is the most expensive neural option.  It covers tasks where the
    transformation is nonlinear (e.g., colour depends on neighbouring cell
    context) and cannot be solved by a single linear conv.

    Architecture:
        Layer 1: Conv2d(C, hidden, ks1×ks1) + bias  - spatial context
        ReLU
        Layer 2: Conv2d(hidden, C,  ks2×ks2)  no bias - colour readout

    Both weight tensors receive L1 pruning (|w| < 0.01 -> 0) before export.
    """
    if not _TORCH_OK:
        return None
    inp, out = _pairs_to_tensors(pairs)
    pad1, pad2 = ks1 // 2, ks2 // 2

    torch.manual_seed(0)
    net = nn.Sequential(
        nn.Dropout(p=0.1),  # <-- NOISE INJECTION: Prevents coordinate memorization
        nn.Conv2d(C,      hidden, kernel_size=ks1, padding=pad1, bias=True),
        nn.ReLU(),
        nn.Conv2d(hidden, C,      kernel_size=ks2, padding=pad2, bias=False),
    )
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_steps
    )

    best_loss  = float("inf")
    best_state = None
    plateau    = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(net(inp), out)
        loss.backward()
        optimizer.step()
        scheduler.step()

        if loss.item() < best_loss - 1e-7:
            best_loss  = loss.item()
            best_state = copy.deepcopy(net.state_dict())
            plateau    = 0
        else:
            plateau += 1

        if step == 300  and best_loss > 0.50: return None
        if step == 800  and best_loss > 0.15: return None
        if plateau > 500 and best_loss > 0.05: return None
        if best_loss < 1e-6: break

    if best_loss > 0.05:
        return None

    net.load_state_dict(best_state)
    
    w1_raw = net[1].weight.detach().numpy() # Index shifted due to Dropout
    b1     = net[1].bias.detach().numpy()
    w2_raw = net[3].weight.detach().numpy() # Index shifted due to Dropout

    X = oh.make_tensor_value_info("input",  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    
    best_model = None
    best_cost  = float("inf")

    # Progressive Pruning Search
    for thresh in [0.2, 0.1, 0.05, 0.01, 0.0]:
        w1_try = np.where(np.abs(w1_raw) < thresh, 0.0, w1_raw)
        w2_try = np.where(np.abs(w2_raw) < thresh, 0.0, w2_raw)
        
        nodes = [
            oh.make_node("Constant", [], ["w1"], value=onh.from_array(w1_try.astype(np.float32), name="w1")),
            oh.make_node("Constant", [], ["b1"], value=onh.from_array(b1.astype(np.float32), name="b1")),
            oh.make_node("Conv", ["input", "w1", "b1"], ["c1"], kernel_shape=[ks1, ks1], pads=[pad1] * 4),
            oh.make_node("Relu", ["c1"], ["r1"]),
            oh.make_node("Constant", [], ["w2"], value=onh.from_array(w2_try.astype(np.float32), name="w2")),
            oh.make_node("Conv", ["r1", "w2"], ["output"], kernel_shape=[ks2, ks2], pads=[pad2] * 4),
        ]
        
        m = oh.make_model(
            oh.make_graph(nodes, "two_layer", [X], [Y]),
            opset_imports=[oh.make_opsetid("", 17)],
        )
        m.ir_version = 8
        
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost:
                best_cost, best_model = c, m
                
    return best_model

# -----------------------------------------------------------------------------
# TIER 3: Three-Layer Convolution Network
# -----------------------------------------------------------------------------

def try_three_layer_conv(pairs, all_pairs, max_steps: int = 3000):
    """
    Train a 3-layer network for the hardest nonlinear spatial tasks.
    Prioritizes representation capacity over MAC cost.
    """
    if not _TORCH_OK:
        return None
    
    inp, out = _pairs_to_tensors(pairs)

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.drop = nn.Dropout(p=0.1) # <-- NOISE INJECTION
            self.c1 = nn.Conv2d(C, 16, 3, padding=1, bias=True)
            self.c2 = nn.Conv2d(16, 16, 3, padding=1, bias=True)
            self.c3 = nn.Conv2d(16, C, 1, bias=True)
        def forward(self, x): 
            return self.c3(torch.relu(self.c2(torch.relu(self.c1(self.drop(x))))))

    torch.manual_seed(0)
    net = Net()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.005)
    best_loss = float("inf")
    best_state = None
    
    for step in range(max_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(net(inp), out)
        loss.backward()
        optimizer.step()
        
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = copy.deepcopy(net.state_dict())
            
        if step == 500 and best_loss > 0.3: 
            return None
        if best_loss < 1e-6: 
            break
            
    if best_loss > 0.05: 
        return None
        
    net.load_state_dict(best_state)

    w1_raw = net.c1.weight.detach().numpy()
    b1     = net.c1.bias.detach().numpy()
    w2_raw = net.c2.weight.detach().numpy()
    b2     = net.c2.bias.detach().numpy()
    w3_raw = net.c3.weight.detach().numpy()
    b3     = net.c3.bias.detach().numpy()

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    
    best_model = None
    best_cost  = float("inf")

    # Progressive Pruning Search
    for thresh in [0.2, 0.1, 0.05, 0.01, 0.0]:
        w1_try = np.where(np.abs(w1_raw) < thresh, 0.0, w1_raw)
        w2_try = np.where(np.abs(w2_raw) < thresh, 0.0, w2_raw)
        w3_try = np.where(np.abs(w3_raw) < thresh, 0.0, w3_raw)
        
        nodes = [
            oh.make_node("Constant", [], ["w1"], value=onh.from_array(w1_try.astype(np.float32), name="w1")),
            oh.make_node("Constant", [], ["b1"], value=onh.from_array(b1.astype(np.float32), name="b1")),
            oh.make_node("Conv", ["input","w1","b1"], ["c1"], kernel_shape=[3,3], pads=[1,1,1,1]),
            oh.make_node("Relu", ["c1"], ["r1"]),
            oh.make_node("Constant", [], ["w2"], value=onh.from_array(w2_try.astype(np.float32), name="w2")),
            oh.make_node("Constant", [], ["b2"], value=onh.from_array(b2.astype(np.float32), name="b2")),
            oh.make_node("Conv", ["r1","w2","b2"], ["c2"], kernel_shape=[3,3], pads=[1,1,1,1]),
            oh.make_node("Relu", ["c2"], ["r2"]),
            oh.make_node("Constant", [], ["w3"], value=onh.from_array(w3_try.astype(np.float32), name="w3")),
            oh.make_node("Constant", [], ["b3"], value=onh.from_array(b3.astype(np.float32), name="b3")),
            oh.make_node("Conv", ["r2","w3","b3"], ["output"], kernel_shape=[1,1], pads=[0,0,0,0]),
        ]
        
        m = oh.make_model(
            oh.make_graph(nodes, "three_layer", [X], [Y]),
            opset_imports=[oh.make_opsetid("", 17)]
        )
        m.ir_version = 8
        
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost:
                best_cost, best_model = c, m
                
    return best_model

print("Neural Waterfall solvers loaded.")

# Cell 9 - The Main Execution Loop: `solve_task` and the Submission Pipeline

## Orchestration Philosophy

The `solve_task` function is the central dispatcher.  Its design follows three
principles:

**1. Validate against everything.**
Every candidate model is checked not just against the training pairs but against
the full union of train, test, and arc-gen examples.  The competition's private
benchmark is explicitly designed to catch models that overfit the small training
set.  Validating against arc-gen (which can contain thousands of examples)
dramatically increases generalisation confidence before any model is accepted.

**2. Keep the cheapest winner.**
Because multiple heuristics can solve the same task (a 180° rotation is both a
double flip and a general spatial permutation), we never stop at the first
correct model.  We store every correct model and compare costs, keeping only the
cheapest.  This is especially important for rotation/flip tasks on small grids,
where a `Slice+Transpose+Pad` chain can be cheaper than a full 900-element
`Gather` index array.

**3. Respect the time budget.**
Each task is allotted a fixed time budget (default: 12 seconds).  Long-running
neural training stages are guarded by `elapsed()` checks so that no single
pathological task starves the remaining 399 tasks.  Early-exit heuristics within
each neural solver also terminate training as soon as it is clear the model is
diverging or stalling.

---

## The Analyse Step

Before running detectors, `analyze_transformation` performs a lightweight
scan of the training pairs to populate a metadata dictionary:

+ **`same_size`** - whether input and output grids share the same shape.
  Many detectors are only applicable when this holds.
+ **`all_same_in_size`** - whether all training inputs share the same shape.
  Gather-based solvers require this to build a consistent index map.
+ **`const_output`** - whether all outputs are identical.  If so, and if
  neural methods also fail, we can fall back to a `make_const_onnx`.
+ **`identity`** - whether the transformation is a no-op.  If detected,
  we immediately return the zero-cost `Identity` ONNX node.
+ **`color_map`** - a global per-colour substitution table derived by
  scanning all pairs.  If consistent, it enables our channel-gather and
  1×1-conv colour-map paths without any spatial computation.

---

## Phase 1 Detector Sequence

The detectors fire in order of typical cheapness:

1. Identity (cost ≈ 0)
2. Colour map / colour swap / colour filter / recolour (0 MACs via channel Gather)
3. Spatial transformations: rotation, flip, transpose, anti-transpose
4. Combined spatial + colour: rotation or flip followed by a colour remap
5. Crop, tile, scale, downscale
6. Pixel permutation (general Gather)
7. Mirror concatenations, quadrant mirror, self-concatenation, rot180 concat
8. Translation, row/col broadcast
9. Border addition, crop-plus-colour
10. Symmetry completion (horizontal, vertical, both axes, diagonal)
11. Zero-ambiguity spatial solver (general case)
12. Relaxed spatial solver (with limited brute-force disambiguation)

---

## Phase 2 Waterfall Sequence

If Phase 1 produces no correct model:

1. Standard Conv 1×1, 3×3, 5×5 (with and without bias)
2. Two-layer Conv (3×3 -> ReLU -> 1×1), hidden=16, 24, 12
3. Three-layer Conv cascade
4. Constant output fallback (last resort when output is fixed)

---

## Submission Assembly

After all 400 tasks are processed, the solved ONNX files are collected and
compressed into `submission.zip` using Python's `zipfile` module with
`ZIP_DEFLATED` compression.  Each file is named `task{NNN}.onnx` to match
the required submission format.  Tasks for which we found no valid solution
are simply omitted (the organisers treat missing entries as unsolved, which
scores 0 rather than a penalty).

A per-task summary line is printed showing the cost, the score under the
competition metric, and the wall-clock solve time.  The final output shows
the aggregate score and total runtime.

In [ ]:
# ==============================================================================
# CELL 10 - ORCHESTRATOR AND SUBMISSION
# ==============================================================================


# -----------------------------------------------------------------------------
# Task metadata analyser
# -----------------------------------------------------------------------------

def analyze_transformation(pairs: list) -> dict:
    """
    Quick pre-scan of training pairs to populate metadata used by detectors.

    Returns a dict with:
        in_sizes        - list of (H, W) tuples for each input grid
        out_sizes       - list of (H, W) tuples for each output grid
        same_size       - True when every pair has matching in/out dimensions
        all_same_in_size - True when all inputs share one shape
        const_output    - True when all outputs are identical
        identity        - True when output == input for every pair
        color_map       - global colour substitution dict (or None)
    """
    in_sizes  = [np.array(p["input"]).shape  for p in pairs]
    out_sizes = [np.array(p["output"]).shape for p in pairs]

    info = {
        "in_sizes":         in_sizes,
        "out_sizes":        out_sizes,
        "same_size":        all(i == o for i, o in zip(in_sizes, out_sizes)),
        "all_same_in_size": len(set(in_sizes)) == 1,
        "const_output":     len({json.dumps(p["output"]) for p in pairs}) == 1,
        "identity":         all(
            shapes_match(p) and np.array_equal(p["input"], p["output"])
            for p in pairs
        ),
    }

    # Build a global colour-substitution table if consistent across all pairs
    if info["same_size"]:
        cmap: dict = {}
        valid = True
        for p in pairs:
            for a, b in zip(np.array(p["input"]).flat,
                            np.array(p["output"]).flat):
                if a in cmap and cmap[a] != b:
                    valid = False; break
                cmap[a] = b
            if not valid:
                break
        info["color_map"] = (
            cmap if valid and any(k != v for k, v in cmap.items()) else None
        )
    else:
        info["color_map"] = None

    return info


# -----------------------------------------------------------------------------
# Main task solver
# -----------------------------------------------------------------------------

def solve_task(task_data: dict,
               task_name:   str   = "",
               time_budget: float = 12.0):
    """
    Orchestrate the full Zero-MAC -> Neural Waterfall pipeline for one task.

    Returns the cheapest correct ONNX model found within the time budget,
    or None if no solution was discovered.
    """
    train_pairs = task_data.get("train", [])
    test_pairs  = task_data.get("test",  [])
    arc_gen     = task_data.get("arc-gen", [])

    if not train_pairs:
        return make_identity_onnx()

    # Validate against every available example for maximum generalisation
    all_pairs = train_pairs + test_pairs + arc_gen
    info      = analyze_transformation(train_pairs)

    best_model = None
    best_cost  = float("inf")
    t_start    = time.time()

    def try_model(model, label: str = "") -> bool:
        nonlocal best_model, best_cost
        if model is None:
            return False
        if check_model_correct(model, all_pairs):
            cost = estimate_model_cost(model)
            if cost < best_cost:
                best_cost  = cost
                best_model = model
            return True
        return False

    def elapsed() -> float:
        return time.time() - t_start

    # -- PHASE 1: ZERO-MAC HEURISTICS -----------------------------------------

    # 1a. Trivial identity
    if info.get("identity"):
        if try_model(make_identity_onnx(), "identity"):
            return best_model

    # Pre-compute spatial flags to inform colour-path decisions
    has_spatial = (
        detect_rotation(train_pairs) is not None
        or detect_flip(train_pairs) is not None
        or detect_transpose(train_pairs)
        or detect_anti_transpose(train_pairs)
    )

    # 1b. Pure colour transformations (same-size pairs, no spatial rearrangement)
    if info.get("color_map") and info["same_size"]:
        cmap = info["color_map"]
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial:
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)),
                      "cmap_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "cmap_conv")

    tc = detect_nonzero_recolor(train_pairs)
    if tc is not None:
        try_model(make_conv1x1_onnx(build_nonzero_recolor_weight(tc)), "nz_recolor")

    tc = detect_bg_recolor(train_pairs)
    if tc is not None:
        try_model(make_conv1x1_onnx(build_bg_recolor_weight(tc)), "bg_recolor")

    repl = detect_color_replace(train_pairs)
    if repl is not None:
        cmap = {i: i for i in range(C)}; cmap.update(repl)
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial:
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)),
                      "repl_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "repl_conv")

    sw = detect_color_swap(train_pairs)
    if sw is not None and not has_spatial:
        a, b = sw
        cmap = {i: i for i in range(C)}; cmap[a] = b; cmap[b] = a
        try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)),
                  "swap_ch")

    cf = detect_color_filter(train_pairs)
    if cf is not None and info["same_size"]:
        cmap = {c_: 0 for c_ in cf}
        cmap.update({c_: c_ for c_ in range(C) if c_ not in cf})
        if can_use_channel_gather_for_cmap(cmap, train_pairs):
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)),
                      "cf_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "cf_conv")

    # 1c. Spatial transformations: rotation, flip, transpose
    rot = detect_rotation(train_pairs)
    if rot is not None:
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_rotation_gather(rot, ih, iw)),
                      f"rot_{rot}_gather")
            if rot == 90:
                try_model(make_cheap_rot90ccw_framed(ih, iw), "rot90ccw_cheap")
            elif rot == 180:
                try_model(make_cheap_rot180_framed(ih, iw), "rot180_cheap")
            elif rot == 270:
                try_model(make_cheap_rot90cw_framed(ih, iw), "rot270cw_cheap")

    flip = detect_flip(train_pairs)
    if flip is not None:
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_flip_gather(flip, ih, iw)),
                      f"flip_{flip}_gather")
            if flip == "vertical":
                try_model(make_cheap_flip_h_framed(ih, iw), "flip_v_cheap")
            elif flip == "horizontal":
                try_model(make_cheap_flip_w_framed(ih, iw), "flip_h_cheap")
            elif flip == "both":
                try_model(make_cheap_rot180_framed(ih, iw), "flip_both_cheap")

    if detect_transpose(train_pairs):
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_cheap_transpose_framed(ih, iw), "transpose_cheap")

    if detect_anti_transpose(train_pairs):
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_cheap_anti_transpose_framed(ih, iw), "anti_tr_cheap")

    # 1d. Combined spatial + colour transformations
    if rot is not None and info.get("color_map"):
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi   = build_rotation_gather(rot, ih, iw)
            cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                try_model(
                    make_spatial_then_channel_gather_onnx(
                        gi, build_channel_gather_indices(cmap)
                    ),
                    "rot_ch_cm",
                )
            try_model(
                make_gather_then_conv1x1_onnx(gi, build_color_map_weight(cmap)),
                "rot_conv_cm",
            )

    if flip is not None and info.get("color_map"):
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi   = build_flip_gather(flip, ih, iw)
            cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                try_model(
                    make_spatial_then_channel_gather_onnx(
                        gi, build_channel_gather_indices(cmap)
                    ),
                    "flip_ch_cm",
                )
            try_model(
                make_gather_then_conv1x1_onnx(gi, build_color_map_weight(cmap)),
                "flip_conv_cm",
            )

    # 1e. Scale transformations
    crop = detect_crop(train_pairs)
    if crop is not None:
        r0, c0, oh_, ow_ = crop
        try_model(make_cheap_crop(r0, c0, oh_, ow_), "crop")

    tile = detect_tile(train_pairs)
    if tile is not None:
        tr, tc = tile
        ih, iw = np.array(train_pairs[0]["input"]).shape
        if ih * tr <= H and iw * tc <= W:
            try_model(make_cheap_tile(tr, tc, ih, iw), "tile")

    sc = detect_scale(train_pairs)
    if sc is not None:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        if ih * sc <= H and iw * sc <= W:
            try_model(make_cheap_upscale(sc, ih, iw), "upscale")

    ds = detect_downscale(train_pairs)
    if ds is not None:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_cheap_downscale(ds, ih, iw), "downscale")

    # 1f. General pixel permutation
    perm = detect_pixel_permutation(train_pairs)
    if perm is not None:
        try_model(make_gather_onnx(perm), "pixel_perm")

    # 1g. Mirror and concatenation patterns
    in_shapes = {np.array(p["input"]).shape for p in train_pairs}
    if len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        if detect_mirror_h_concat(train_pairs) and iw * 2 <= W:
            try_model(make_cheap_concat_flip_w(ih, iw), "mirror_h")
        if detect_mirror_v_concat(train_pairs) and ih * 2 <= H:
            try_model(make_cheap_concat_flip_h(ih, iw), "mirror_v")
        if detect_self_concat_h(train_pairs) and iw * 2 <= W:
            try_model(make_cheap_self_concat_w(ih, iw), "self_concat_h")
        if detect_self_concat_v(train_pairs) and ih * 2 <= H:
            try_model(make_cheap_self_concat_h(ih, iw), "self_concat_v")
        qm = detect_quad_mirror(train_pairs)
        if qm is not None and ih * 2 <= H and iw * 2 <= W:
            try_model(make_cheap_quadrant_mirror(ih, iw), "quad_mirror")
        if detect_concat_rot180_w(train_pairs) and iw * 2 <= W:
            try_model(make_cheap_concat_rot180_w(ih, iw), "concat_rot180_w")

    # 1h. Translation
    trans = detect_translation(train_pairs)
    if trans is not None:
        dr, dc = trans
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            m = make_cheap_framed_translation(ih, iw, dr, dc)
            if m:
                try_model(m, "translate")

    # 1i. Row / col broadcast
    in_shapes = {np.array(p["input"]).shape for p in train_pairs}
    if len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        rb = detect_row_broadcast(train_pairs)
        if rb is not None:
            try_model(make_cheap_row_broadcast(ih, iw, rb), "row_broadcast")
        cb = detect_col_broadcast(train_pairs)
        if cb is not None:
            try_model(make_cheap_col_broadcast(ih, iw, cb), "col_broadcast")

    # 1j. Border addition
    ba = detect_border_add(train_pairs)
    if ba is not None:
        b_width, fill = ba
        in_shapes = {np.array(p["input"]).shape for p in train_pairs}
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            if ih + 2 * b_width <= H and iw + 2 * b_width <= W:
                m = make_cheap_border_add(ih, iw, b_width, fill)
                if m:
                    try_model(m, "border_add")

    # 1k. Crop + colour remap
    ccp = detect_crop_plus_color(train_pairs)
    if ccp is not None:
        r0, c0, oh_, ow_, cm = ccp
        try_model(make_cheap_crop_then_color(r0, c0, oh_, ow_, cm), "crop_color")

    # 1l. Symmetry completion
    if info["same_size"] and info["all_same_in_size"]:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        if detect_h_symmetry_complete(train_pairs):
            try_model(
                make_symmetry_completion_onnx(build_h_sym_complete_gathers(ih, iw)),
                "h_sym",
            )
        if detect_v_symmetry_complete(train_pairs):
            try_model(
                make_symmetry_completion_onnx(build_v_sym_complete_gathers(ih, iw)),
                "v_sym",
            )
        if detect_both_symmetry_complete(train_pairs):
            try_model(
                make_symmetry_completion_onnx(
                    build_both_sym_complete_gathers(ih, iw)
                ),
                "both_sym",
            )
        if ih == iw and detect_diag_symmetry_complete(train_pairs):
            g = build_diag_sym_complete_gathers(ih, iw)
            if g:
                try_model(make_symmetry_completion_onnx(g), "diag_sym")

    # 1m. General spatial solvers
    m = try_spatial_zero_ambiguity(all_pairs)
    if m:
        try_model(m, "spatial_unambig")

    if best_model is None and elapsed() < time_budget:
        m = try_spatial_relaxed(all_pairs, max_ambig=12)
        if m:
            try_model(m, "spatial_relaxed")

    if best_model is not None:
        return best_model

    # -- PHASE 2: NEURAL WATERFALL ---------------------------------------------

    # Tier 1: Standard Conv
    for ks in [1, 3, 5]:
        if elapsed() > time_budget:
            break
        m = try_standard_conv_fast(
            train_pairs, all_pairs, kernel_size=ks, use_bias=False
        )
        if try_model(m, f"std_conv_k{ks}"):
            break
        m = try_standard_conv_fast(
            train_pairs, all_pairs, kernel_size=ks, use_bias=True
        )
        if try_model(m, f"std_conv_k{ks}_bias"):
            break

    # Tier 2: Two-layer ReLU network
    if best_model is None:
        for ks1, ks2, hid in [(3, 1, 16), (3, 1, 24), (5, 1, 12)]:
            if elapsed() > time_budget:
                break
            m = try_two_layer_conv_fast(
                train_pairs, all_pairs, ks1=ks1, ks2=ks2, hidden=hid
            )
            if try_model(m, f"two_layer_{ks1}_{hid}"):
                break

    # Tier 3: Three-layer ReLU network (Highest capacity)
    if best_model is None:
        if elapsed() < time_budget:
            m = try_three_layer_conv(train_pairs, all_pairs)
            try_model(m, "three_layer")

    # Final safety net: constant output (only valid when output never varies)
    if best_model is None and info.get("const_output"):
        try_model(
            make_const_onnx(grid_to_tensor(train_pairs[0]["output"])),
            "const_output",
        )

    return best_model


# -----------------------------------------------------------------------------
# Persistence helper
# -----------------------------------------------------------------------------

def save_model(model, path: Path) -> None:
    """Serialise an ONNX model to disk."""
    with open(path, "wb") as f:
        f.write(model.SerializeToString())


# -----------------------------------------------------------------------------
# Main execution loop
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    # Locate task JSON files
    task_files = sorted(TASK_DIR.glob("task*.json"))
    if not task_files:
        for d in [Path("/kaggle/input"), Path(".")]:
            if d.exists():
                task_files = sorted(d.glob("task*.json"))
                if task_files:
                    break

    print(f"NeuroGolf 2026 | Found {len(task_files)} task(s) in: {TASK_DIR}")
    if not task_files:
        sys.exit(1)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    solved      = []
    onnx_files  = []
    total_score = 0.0
    wall_start  = time.time()

    for tp in task_files:
        tn   = tp.stem
        tnum = tn.replace("task", "")

        try:
            td = load_task(tp)
        except Exception as e:
            print(f"  {tn}: LOAD ERROR  {e}")
            continue

        t0 = time.time()
        try:
            model = solve_task(td, tn, time_budget=12.0)
        except Exception as e:
            print(f"  {tn}: SOLVE ERROR  {e}")
            continue

        dt = time.time() - t0

        if model is None:
            print(f"  {tn}: no solution ({dt:.1f}s)")
            continue

        op = OUTPUT_DIR / f"task{tnum}.onnx"
        save_model(model, op)
        onnx_files.append(op)

        cost  = estimate_model_cost(model)
        score = max(1.0, 25.0 - math.log(max(1, cost)))
        total_score += score
        solved.append(tn)

        print(
            f"  {tn}  cost={cost:>10,}  score={score:5.2f}  time={dt:.1f}s"
        )

    wall_time = time.time() - wall_start
    print(
        f"\nSolved {len(solved)}/{len(task_files)} tasks  |  "
        f"Total score = {total_score:.1f}  |  "
        f"Wall time = {wall_time:.0f}s"
    )

    if onnx_files:
        zip_path = OUTPUT_DIR / "submission.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in onnx_files:
                zf.write(f, f.name)
        print(
            f"\nCreated submission.zip  ({zip_path.stat().st_size / 1024:.1f} KB)  "
            f"containing {len(onnx_files)} ONNX file(s)."
        )
    else:
        print("\nNo ONNX files produced.  submission.zip not created.")